## Filtering data using a CTE


### Business requirement: Extract successful transactions whose amounts exceed ₦30,000.

In [ ]:
WITH
    successful_transactions AS (
        SELECT
            transaction_id,
            customer_id,
            amount,
            status
        FROM payment_transactions
        WHERE
            status = 'Success'
    )
SELECT
    transaction_id,
    customer_id,
    amount,
    status
FROM successful_transactions
WHERE
    amount > 30000
ORDER BY amount DESC;

## CTE with aggregation

### Business requirement: Calculate total successful spending per customer ID and identify those who have spent at least ₦50,000.

In [ ]:
WITH
    customer_spending AS (
        SELECT
            customer_id,
            COUNT(*) AS transaction_count,
            SUM(amount) AS total_spent
        FROM pament_transactions
        WHERE
            status = 'Success'
        GROUP BY
            customer_id
    )
SELECT
    customer_id,
    transaction_count,
    total_spent
FROM customer_spending
WHERE
    total_spent >= 50000
ORDER BY total_spent DESC;

## CTE with JOIN

### Business requirement: Calculate successful spending per customer and enrich the results with customer information. Preserve every registered customer, including customers with zero successful transactions.

In [ ]:
WITH
    customer_spending AS (
        SELECT
            customer_id,
            COUNT(*) AS total_transactions,
            SUM(amount) AS total_spent
        FROM payment_transactions
        WHERE
            status = 'Success'
        GROUP BY
            customer_id
    )
SELECT
    c.customer_id,
    c.customer_name,
    c.country,
    COALESCE(s.total_transactions, 0) AS total_transactions,
    COALESCE(s.total_spent, 0) AS total_spent
FROM
    customer c
    LEFT JOIN customer_spending AS s ON c.customer_id = s.customer_id
ORDER BY s.total_spent DESC, c.customer_id ASC;

## Multiple dependent CTEs

### Business requirement: Classify registered customers based on their total successful spending.

In [ ]:
WITH successful_transactions AS (
    SELECT transaction_id,
              customer_id,
              amount,
              status
    FROM payment_transactions
    WHERE status = 'Success'
)

WITH customer_spending AS (
    SELECT customer_id,
        COUNT(*) AS total_transactions,
        SUM(amount) AS total_spent,
    FROM successful_transactions
    GROUP BY customer_id
)

WITH customer_segment AS (
    SELECT customer_id,
        total_transactions,
        total_spent,
        CASE
        WHEN total_spent >= 100000 THEN 'Premium' 
        WHEN total_spent >= 50000 THEN 'Standard'
        ELSE 'Basic'
        END AS customer_segment     
)

SELECT c.customer_id,
    c.customer_name,
    s.total_transaction,
    s.total_spent
    s.customer_segment
    FROM customer c
    INNER JOIN customer_segment AS s 
    ON c.customer_id = s.customer_id
    ORDER BY s.total_spent DESC, s.customer_id ASC;